# Gradient Boosting

- Gradient-boosted trees consists in an algorithm that ensembles the predictions of weker prediction models, particularly decision trees. Each of these predictors will correct their predecessor. Unlike many algorithms, this one doesn't just deal with improving weights at each iteration, instead tries to learn frrom the previous predictors residual errors and fit the following predictor to those.
- This algorithm can be seen as an optimization one bassed on a certain loss function. 

## Gradient Tree Boosting (GTB) in Scikit learn

- Scikit-learn (0.21) provides us with two options to implement this algorithm (they are availabl both for classification and regression, but we will only focus on the regression ones), both of them inspired by LightGBM:
    - __'HistGradientBoostingRegressor'__ and
    - __'GradientBoostingRegressor'__
- An interesting detail about this implementations is that they support missing values and for the __'HistGradientBoostingRegressor'__ one categorical data is supported aswell. 
- For small datasets, the use of bins might contribute to very close split points, ence why, in this case, __'GradientBoostingRegressor'__ is preferible. However, our dataset granularity is moderatly high and, as indicated by the official documentation, for a sample with tens of thousands of samples, __'HistGradientBoostingRegressor'__ is preferible. 
- In addition, __'HistGradientBoostingRegressor'__ has the advantage of increased time efficiency and complexity (that will be later on explained in the notebook) compared to __'GradientBoostingRegressor'__. However, the former does not natively support some features of the latter. 

Note: all of these informations are available at the scikit learn documentation.

## 2. Understanding __'HistGradientBoostingRegressor'__

- We decided to use this algorithm because of the infra citated reasons as well as because of the reasons citated in ths section. 

### 2.1. Further Investigating why this approch is faster

### 2.2. Missing Values and Categorical Features Support 

### 2.3. Sample Weights

### 2.4. Parameters

### 2.5. Constraints

## 3. Setup and Data Loading

In [4]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error

In [5]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error

# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# lists previously defined in 05_0_preproc_helpers.ipynb
numeric_features = num_feat                      # ['year', 'mileage', ...]
categorical_features = cat_feat                  # ['Brand', 'model', 'transmission', 'fuelType']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)


X shape: (75973, 11)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'paintQuality%', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


## 4. Randomized Hyperparameter Search with K-Fold Cross-Validation

In [6]:
# ---------------------------------------------------------
# 2. CONFIGURAÇÃO DO CROSS-VALIDATION
# ---------------------------------------------------------
N_SPLITS = 8 
RANDOM_STATE = 42

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# ---------------------------------------------------------
# 3. HIPERPARÂMETROS (GRELHA PARA HIST GRADIENT BOOSTING)
# ---------------------------------------------------------
param_distributions = {
    # Número de iterações (equivalente a n_estimators)
    "max_iter": [600, 800, 1000],
    
    # Velocidade de aprendizagem (CRÍTICO: Menor = Mais preciso, mas precisa de mais iterações)
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    
    # Complexidade da árvore (O HGB prefere controlar por 'leaf_nodes' em vez de profundidade)
    "max_leaf_nodes": [31, 63, 127], # 31 é o padrão estilo LightGBM
    "max_depth": [None, 10, 20, 11, 15],     # None deixa crescer até atingir o limite de folhas
    
    # Regularização (Para evitar overfitting, já que não temos feature selection aqui)
    "l2_regularization": [0, 1, 10],
    
    # Minimo de amostras por folha (Robustez)
    "min_samples_leaf": [20, 40]
}

# ---------------------------------------------------------
# 4. CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
N_RANDOM_CONFIGS = 25

sampler = ParameterSampler(
    param_distributions, 
    n_iter=N_RANDOM_CONFIGS, 
    random_state=RANDOM_STATE 
)

search_results = [] 
best_rmse = np.inf 
best_config = None 

log_path = "hgb_first_approach_log.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg) 
        log_file.write(msg + "\n") 
        log_file.flush() 

    log("# =============================")
    log(f"# START OF HIST GRADIENT BOOSTING (NO FEATURE SELECTION)")
    log(f"# {N_RANDOM_CONFIGS} ITERATIONS")
    log("# =============================")
    
    # -----------------------------
    # LOOP PRINCIPAL
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        # Listas para métricas
        fold_rmses, fold_maes, fold_r2s = [], [], []
        fold_rmses_train, fold_maes_train = [], []
        fold_med_ae, fold_mape, fold_bias = [], [], []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            
            # --- A) Split Data ---
            X_train = X.iloc[train_idx].copy()
            X_val   = X.iloc[val_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_val   = y.iloc[val_idx].copy()

            log(f"\n[C{config_id}|F{fold}] Processing fold...")

            # --- B) Pipeline de Limpeza (Igual ao Random Forest) ---
            # B.1 Numerics
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            tax_state = fit_tax_imputer(X_train, tax_col="tax", do_abs=True)
            X_train = transform_tax_imputer(X_train, state=tax_state)
            X_val   = transform_tax_imputer(X_val,   state=tax_state)

            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            paint_state = fit_paint_quality_imputer(X_train, paint_col="paintQuality%")
            X_train = transform_paint_quality_imputer(X_train, state=paint_state)
            X_val   = transform_paint_quality_imputer(X_val,   state=paint_state)

            owners_state = fit_previous_owners_imputer(
                X_train, owners_col="previousOwners", year_col="year", mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            # B.2 Categorical Resolvers
            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands=valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand=valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions=valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)
            
            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes=valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

            # --- C) Encoding (Target + OneHot) ---
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # --- D) Matriz Final (SEM SCALING) ---
            # O HistGradientBoosting não precisa de scaling
            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)
            
            # --- E) Treino do Modelo ---
            hgb_model = HistGradientBoostingRegressor(
                random_state=RANDOM_STATE,
                **params 
            )
            hgb_model.fit(X_train_final, y_train)

            # --- F) Previsões e Métricas ---
            y_pred_train = hgb_model.predict(X_train_final)
            y_pred_val   = hgb_model.predict(X_val_final)

            # Standard Metrics
            mse_val, mae_val = mean_squared_error(y_val, y_pred_val), mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            r2_val = r2_score(y_val, y_pred_val)
            
            mse_tr, mae_tr = mean_squared_error(y_train, y_pred_train), mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            
            # Robust Metrics
            med_ae_val = median_absolute_error(y_val, y_pred_val)
            mape_val = mean_absolute_percentage_error(y_val, y_pred_val) * 100 
            bias_val = np.mean(y_val - y_pred_val)

            log(f"  > [TRAIN] RMSE: {rmse_tr:.2f} | [VAL] RMSE: {rmse_val:.2f}, MedAE: {med_ae_val:.2f}")

            # Store per fold
            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)
            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_med_ae.append(med_ae_val)
            fold_mape.append(mape_val)
            fold_bias.append(bias_val)

        # Average metrics across folds
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_med_ae_val = np.mean(fold_med_ae)
        mean_bias_val = np.mean(fold_bias)
        
        # Combined score for ranking
        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - [VAL AVG] RMSE: {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | MedAE: {mean_med_ae_val:.4f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "med_ae_mean": mean_med_ae_val,
            "bias_mean": mean_bias_val,
            "combo_score": combo_score,
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

    log("# END OF HGB SEARCH")

# -----------------------------
# Results
# -----------------------------
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True)

print("\nTop 5 Configurations by RMSE:")
cols = ["config_id", "learning_rate", "max_iter", "max_leaf_nodes", "rmse_mean", "mae_mean", "med_ae_mean"]
display(results_df_sorted[cols].head(5))

print("\nBest Config found:")
print(best_config)

# =============================
# START OF HIST GRADIENT BOOSTING (NO FEATURE SELECTION)
# 25 ITERATIONS
# =============================

######## CONFIG 1/25 ########
Parameters: {'min_samples_leaf': 40, 'max_leaf_nodes': 63, 'max_iter': 800, 'max_depth': 11, 'learning_rate': 0.01, 'l2_regularization': 1}

[C1|F1] Processing fold...
  > [TRAIN] RMSE: 2330.15 | [VAL] RMSE: 2237.24, MedAE: 971.58

[C1|F2] Processing fold...
  > [TRAIN] RMSE: 2258.67 | [VAL] RMSE: 2371.45, MedAE: 991.23

[C1|F3] Processing fold...
  > [TRAIN] RMSE: 2249.32 | [VAL] RMSE: 2512.56, MedAE: 982.23

[C1|F4] Processing fold...
  > [TRAIN] RMSE: 2193.58 | [VAL] RMSE: 2817.93, MedAE: 988.96

[C1|F5] Processing fold...
  > [TRAIN] RMSE: 2248.39 | [VAL] RMSE: 2370.59, MedAE: 958.12

[C1|F6] Processing fold...
  > [TRAIN] RMSE: 2255.68 | [VAL] RMSE: 2437.33, MedAE: 972.95

[C1|F7] Processing fold...
  > [TRAIN] RMSE: 2260.88 | [VAL] RMSE: 2281.21, MedAE: 985.59

[C1|F8] Processing fold...
  > [TRAIN] RMSE: 2267.08 |

,config_id,learning_rate,max_iter,max_leaf_nodes,rmse_mean,mae_mean,med_ae_mean
5,6,0.10,600,127,2152.832645,1297.001738,835.828937
13,14,0.05,1000,63,2155.058976,1316.300131,860.028244
1,2,0.10,800,63,2164.960756,1312.920926,851.992113
10,11,0.20,600,127,2173.897177,1311.735929,848.195958
12,13,0.20,600,127,2185.327233,1317.107375,851.002171



Best Config found:
{'min_samples_leaf': 20, 'max_leaf_nodes': 127, 'max_iter': 600, 'max_depth': 15, 'learning_rate': 0.1, 'l2_regularization': 1}


In [ ]:
# ==============================================================================
# 1. CONFIGURAÇÃO FINAL (HistGradientBoosting)
# ==============================================================================
# Parâmetros sem Feature Selection
final_config = {
    'min_samples_leaf': 20, 
    'max_leaf_nodes': 127, 
    'max_iter': 600, 
    'max_depth': 15, 
    'learning_rate': 0.1, 
    'l2_regularization': 1
}

print(f"Config final HGB: {final_config}")
print("Feature Selection: DESATIVADO (Usando todas as colunas)")

# ==============================================================================
# 2. PREPARAÇÃO E LIMPEZA INICIAL (X_FULL)
# ==============================================================================
X_full = X.copy()
y_full = y.copy()

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

# 2.1 Normalização
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])
        X_full = column_string_transformer( 
            X_full, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )

# 2.2 Correção Inicial de Marcas
invalids = sorted([b for b in X_full['Brand'].unique() if b not in valid_brands], key=len)
X_full, corrections, remaining_invalids = correct_invalid_brands_in_df(
    X_full, col='Brand', valid_brands=valid_brands, invalids=invalids
)
print(f"Correções de Brand iniciais: {len(corrections)}")

high_card_features = ["Brand", "model"]  
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# ==============================================================================
# 3. PIPELINE ROBUSTO (FIT & TRANSFORM)
# ==============================================================================

# --- A) Numéricas ---
year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

mileage_state = fit_mileage_imputer(X_full, mileage_col="mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

tax_state = fit_tax_imputer(X_full, tax_col="tax", do_abs=True)
X_full = transform_tax_imputer(X_full, state=tax_state)

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

paint_state = fit_paint_quality_imputer(X_full, paint_col="paintQuality%")
X_full = transform_paint_quality_imputer(X_full, state=paint_state)

owners_state = fit_previous_owners_imputer(
    X_full, owners_col="previousOwners", year_col="year", mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

# --- B) Resolvers Categóricos ---
brand_state = fit_ambiguous_brand_resolver(X_full, valid_brands=valid_brands)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(X_full, valid_models_by_brand=valid_models_by_brand)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(X_full, valid_transmissions=valid_transmissions)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(X_full, valid_fueltypes=valid_fueltypes)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# ==============================================================================
# 4. ENCODING (Sem Scaling)
# ==============================================================================

te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])
X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

# Concatenar numéricas + categóricas
X_full_final = pd.concat([X_full[numeric_features], X_full_cat], axis=1)
print("Shape final para treino:", X_full_final.shape)

# ==============================================================================
# 5. TREINO DO MODELO FINAL (HIST GRADIENT BOOSTING)
# ==============================================================================
hgb_final = HistGradientBoostingRegressor(
    random_state=RANDOM_STATE,
    **final_config 
)

hgb_final.fit(X_full_final, y_full)
print("Modelo HGB final treinado com sucesso.")

# ==============================================================================
# 6. PROCESSAR O TEST SET (TRANSFORM ONLY)
# ==============================================================================
test_df = pd.read_csv("../../project_data/test.csv")

for col in cols_to_normalize:
    if col in test_df.columns:
        test_df[col] = fill_unknown(test_df[col])
        test_df = column_string_transformer(
            test_df, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )

X_test = test_df[numeric_features + categorical_features].copy()

# 6.1 Transforms
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
X_test = transform_paint_quality_imputer(X_test, state=paint_state)
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# 6.2 Resolvers
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# 6.3 Encoding
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

# 6.4 Matriz Final Teste
X_test_final = pd.concat([X_test[numeric_features], X_test_cat], axis=1)
print("Shape final do Test Set:", X_test_final.shape)

# ==============================================================================
# 7. PREVISÃO E SUBMISSÃO
# ==============================================================================
y_test_pred = hgb_final.predict(X_test_final)

# Arredondar para inteiro
y_test_pred_round = np.round(y_test_pred).astype(int)

submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred
})

submission_path = "hgb_final_submission_no_fs.csv" # version 15
submission.to_csv(submission_path, index=False)

print(f"Submissão guardada em: {submission_path}")

Config final HGB: {'min_samples_leaf': 20, 'max_leaf_nodes': 127, 'max_iter': 600, 'max_depth': 15, 'learning_rate': 0.1, 'l2_regularization': 1}
Feature Selection: DESATIVADO (Usando todas as colunas)
Correções de Brand iniciais: 0
Shape final para treino: (75973, 17)
Modelo HGB final treinado com sucesso.
Shape final do Test Set: (32567, 17)
Submissão guardada em: hgb_final_submission_no_fs.csv


In [11]:
# ==============================================================================
# 1. CONFIGURAÇÃO FINAL (HistGradientBoosting)
# ==============================================================================
# Os teus parâmetros escolhidos:
final_config = {
    'min_samples_leaf': 20, 
    'max_leaf_nodes': 127, 
    'max_iter': 600, 
    'max_depth': 15, 
    'learning_rate': 0.1, 
    'l2_regularization': 1,
    
    # Percentagem de features a manter (Assumi 0.9, ajusta se necessário)
    'n_features_to_select': 0.9 
}

# Retirar o parâmetro do seletor para não ir para o modelo HGB
n_features_pct = final_config.pop("n_features_to_select", 1.0) 

print(f"Config final HGB: {final_config}")
print(f"Features a manter: {n_features_pct * 100}%")

# ==============================================================================
# 2. PREPARAÇÃO E LIMPEZA INICIAL (X_FULL)
# ==============================================================================
X_full = X.copy()
y_full = y.copy()

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

# 2.1 Normalização
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])
        X_full = column_string_transformer( 
            X_full, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )

# 2.2 Correção Inicial de Marcas
invalids = sorted([b for b in X_full['Brand'].unique() if b not in valid_brands], key=len)
X_full, corrections, remaining_invalids = correct_invalid_brands_in_df(
    X_full, col='Brand', valid_brands=valid_brands, invalids=invalids
)
print(f"Correções de Brand iniciais: {len(corrections)}")

high_card_features = ["Brand", "model"]  
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# ==============================================================================
# 3. PIPELINE ROBUSTO (FIT & TRANSFORM)
# ==============================================================================

# --- A) Numéricas ---
year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

mileage_state = fit_mileage_imputer(X_full, mileage_col="mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

tax_state = fit_tax_imputer(X_full, tax_col="tax", do_abs=True)
X_full = transform_tax_imputer(X_full, state=tax_state)

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

paint_state = fit_paint_quality_imputer(X_full, paint_col="paintQuality%")
X_full = transform_paint_quality_imputer(X_full, state=paint_state)

owners_state = fit_previous_owners_imputer(
    X_full, owners_col="previousOwners", year_col="year", mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

# --- B) Resolvers Categóricos ---
brand_state = fit_ambiguous_brand_resolver(X_full, valid_brands=valid_brands)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(X_full, valid_models_by_brand=valid_models_by_brand)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(X_full, valid_transmissions=valid_transmissions)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(X_full, valid_fueltypes=valid_fueltypes)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# ==============================================================================
# 4. ENCODING (Sem Scaling, HGB não precisa)
# ==============================================================================

te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])
X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

# Concatenar numéricas (sem scaler) + categóricas
X_full_final = pd.concat([X_full[numeric_features], X_full_cat], axis=1)
print("Shape antes da seleção:", X_full_final.shape)

# ==============================================================================
# 5. FEATURE SELECTION (Usando RF como 'Scout')
# ==============================================================================
selector = None 

if n_features_pct < 1.0 and n_features_pct is not None:
    total_cols = X_full_final.shape[1]
    n_feats_final = int(total_cols * n_features_pct)
    n_feats_final = max(1, n_feats_final)
    
    print(f"A selecionar as melhores {n_feats_final} features...")
    
    # Usamos Random Forest para selecionar (funciona muito bem com HGB)
    selector = MyRandomForestSelector(
        n_features=n_feats_final, 
        n_estimators=50, 
        random_state=RANDOM_STATE
    )
    
    selector.fit(X_full_final, y_full)
    X_full_final = selector.transform(X_full_final)
    
    print("Shape DEPOIS da seleção:", X_full_final.shape)
else:
    print("Feature Selection ignorado.")

# ==============================================================================
# 6. TREINO DO MODELO FINAL (HIST GRADIENT BOOSTING)
# ==============================================================================
hgb_final = HistGradientBoostingRegressor(
    random_state=RANDOM_STATE,
    **final_config 
)

hgb_final.fit(X_full_final, y_full)
print("Modelo HGB final treinado com sucesso.")

# ==============================================================================
# 7. PROCESSAR O TEST SET (TRANSFORM ONLY)
# ==============================================================================
test_df = pd.read_csv("../../project_data/test.csv")

for col in cols_to_normalize:
    if col in test_df.columns:
        test_df[col] = fill_unknown(test_df[col])
        test_df = column_string_transformer(
            test_df, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )

X_test = test_df[numeric_features + categorical_features].copy()

# 7.1 Transforms
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
X_test = transform_paint_quality_imputer(X_test, state=paint_state)
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# 7.2 Resolvers
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# 7.3 Encoding
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

# 7.4 Matriz Final Teste (Sem scaler)
X_test_final = pd.concat([X_test[numeric_features], X_test_cat], axis=1)

# 7.5 FEATURE SELECTION
if selector is not None:
    X_test_final = selector.transform(X_test_final)
    print("Seleção de features aplicada ao Test Set.")

print("Shape final do Test Set:", X_test_final.shape)

# ==============================================================================
# 8. PREVISÃO E SUBMISSÃO
# ==============================================================================
y_test_pred = hgb_final.predict(X_test_final)

# Arredondar para inteiro
y_test_pred_round = np.round(y_test_pred).astype(int)

submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred
})

submission_path = "hgb_final_submission_1.csv" #version 14
submission.to_csv(submission_path, index=False)

print(f"Submissão guardada em: {submission_path}")

Config final HGB: {'min_samples_leaf': 20, 'max_leaf_nodes': 127, 'max_iter': 600, 'max_depth': 15, 'learning_rate': 0.1, 'l2_regularization': 1}
Features a manter: 90.0%
Correções de Brand iniciais: 0
Shape antes da seleção: (75973, 17)
A selecionar as melhores 15 features...
Shape DEPOIS da seleção: (75973, 15)
Modelo HGB final treinado com sucesso.
Seleção de features aplicada ao Test Set.
Shape final do Test Set: (32567, 15)
Submissão guardada em: hgb_final_submission_1.csv
